# t检验完整实现示例

## 目标

掌握各种t检验方法的应用，包括单样本t检验、独立双样本t检验和配对样本t检验。

## 内容

- 单样本t检验：检验样本均值是否等于某个已知值
- 独立双样本t检验：比较两个独立样本的均值
- 配对样本t检验：比较同一组对象在不同条件下的差异
- 效应量计算：Cohen's d
- 可视化：t分布、置信区间、数据分布

## 1. 数据准备

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 设置随机种子保证可重复性
np.random.seed(42)

print("库导入完成")

## 2. t分布可视化

t分布与正态分布的对比

In [ ]:
# 绘制不同自由度的t分布
plt.figure(figsize=(12, 6))

x = np.linspace(-4, 4, 1000)
degrees_of_freedom = [1, 3, 5, 10, 30, 100]

# 标准正态分布
plt.plot(x, stats.norm.pdf(x, 0, 1), 'k-', linewidth=3, label='标准正态分布')

# 不同自由度的t分布
colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown']
for i, df in enumerate(degrees_of_freedom):
    plt.plot(x, stats.t.pdf(x, df), color=colors[i], linewidth=1.5, label=f't分布 (df={df})')

plt.xlabel('值')
plt.ylabel('概率密度')
plt.title('t分布与标准正态分布的对比')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("观察：自由度越大，t分布越接近标准正态分布")

## 3. 单样本t检验

### 3.1 检验示例

某工厂声称其生产的电池平均寿命为100小时。随机抽取25个电池，测得平均寿命为98小时，标准差为5小时。

In [ ]:
# 生成样本数据
n = 25
mu_hypothesized = 100
sample_mean = 98
sample_std = 5

# 生成符合描述的样本
sample = np.random.normal(sample_mean, sample_std, n)

# 执行单样本t检验
t_stat, p_value = stats.ttest_1samp(sample, mu_hypothesized)

print("单样本t检验结果：")
print(f"假设的总体均值: {mu_hypothesized}")
print(f"样本均值: {np.mean(sample):.2f}")
print(f"样本标准差: {np.std(sample, ddof=1):.2f}")
print(f"样本量: {n}")
print(f"\nt统计量: {t_stat:.4f}")
print(f"p值: {p_value:.4f}")

# 决策
alpha = 0.05
if p_value < alpha:
    print(f"\n由于 p = {p_value:.4f} < {alpha}，拒绝原假设")
    print("结论：电池的平均寿命与工厂声称的100小时有显著差异")
else:
    print(f"\n由于 p = {p_value:.4f} >= {alpha}，不拒绝原假设")
    print("结论：没有足够证据表明电池的平均寿命与工厂声称的100小时有差异")

### 3.2 单样本t检验可视化

In [ ]:
# 计算置信区间
confidence_level = 0.95
alpha = 1 - confidence_level
t_critical = stats.t.ppf(1 - alpha/2, df=n-1)
margin_of_error = t_critical * (np.std(sample, ddof=1) / np.sqrt(n))
ci_lower = np.mean(sample) - margin_of_error
ci_upper = np.mean(sample) + margin_of_error

print(f"{confidence_level*100}% 置信区间: [{ci_lower:.2f}, {ci_upper:.2f}]")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：数据分布
axes[0].hist(sample, bins=10, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].axvline(np.mean(sample), color='red', linewidth=2, linestyle='--', label=f'样本均值 = {np.mean(sample):.2f}')
axes[0].axvline(mu_hypothesized, color='green', linewidth=2, linestyle='-', label=f'假设均值 = {mu_hypothesized}')
axes[0].set_xlabel('电池寿命（小时）')
axes[0].set_ylabel('频数')
axes[0].set_title('电池寿命分布')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 右图：t分布和p值区域
df = n - 1
x = np.linspace(-4, 4, 1000)
y = stats.t.pdf(x, df)

axes[1].plot(x, y, 'b-', linewidth=2, label=f't分布 (df={df})')
axes[1].fill_between(x, y, where=(x <= -abs(t_stat)) | (x >= abs(t_stat)), alpha=0.3, color='red', label=f'p值区域')
axes[1].axvline(t_stat, color='red', linestyle='--', linewidth=2)
axes[1].axvline(-t_stat, color='red', linestyle='--', linewidth=2)
axes[1].axvline(stats.t.ppf(1-alpha/2, df), color='green', linestyle=':', linewidth=1.5, label=f'临界值 (α={alpha})')
axes[1].axvline(-stats.t.ppf(1-alpha/2, df), color='green', linestyle=':', linewidth=1.5)
axes[1].set_xlabel('t值')
axes[1].set_ylabel('概率密度')
axes[1].set_title(f't分布与检验结果 (t={t_stat:.2f}, p={p_value:.4f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. 独立双样本t检验

### 4.1 等方差假设（合并方差t检验）

比较两种教学方法的效果。

In [ ]:
# 生成两种教学方法的数据
np.random.seed(42)
n1 = 30
n2 = 30

# 方法A：均值较高
group_A = np.random.normal(85, 6, n1)

# 方法B：均值稍低
group_B = np.random.normal(82, 6, n2)

# 等方差t检验
t_stat_equal_var, p_value_equal_var = stats.ttest_ind(group_A, group_B, equal_var=True)

print("独立双样本t检验（等方差假设）")
print("=" * 40)
print(f"方法A - 均值: {np.mean(group_A):.2f}, 标准差: {np.std(group_A, ddof=1):.2f}, 样本量: {n1}")
print(f"方法B - 均值: {np.mean(group_B):.2f}, 标准差: {np.std(group_B, ddof=1):.2f}, 样本量: {n2}")
print(f"\nt统计量: {t_stat_equal_var:.4f}")
print(f"p值: {p_value_equal_var:.4f}")

# 决策
alpha = 0.05
if p_value_equal_var < alpha:
    print(f"\n结论：拒绝原假设，两种方法的效果有显著差异")
else:
    print(f"\n结论：不拒绝原假设，没有证据表明两种方法的效果有差异")

### 4.2 异方差假设（Welch's t-test）

In [ ]:
# 生成方差不等的样本
np.random.seed(42)

# 组1：小方差
group1 = np.random.normal(75, 3, 25)

# 组2：大方差
group2 = np.random.normal(78, 8, 25)

# 比较两种t检验方法
t_stat_equal, p_equal = stats.ttest_ind(group1, group2, equal_var=True)
t_stat_welch, p_welch = stats.ttest_ind(group1, group2, equal_var=False)

print("等方差 vs Welch t检验对比")
print("=" * 40)
print(f"组1 - 均值: {np.mean(group1):.2f}, 标准差: {np.std(group1, ddof=1):.2f}")
print(f"组2 - 均值: {np.mean(group2):.2f}, 标准差: {np.std(group2, ddof=1):.2f}")
print(f"\n等方差t检验: t = {t_stat_equal:.4f}, p = {p_equal:.4f}")
print(f"Welch t检验:  t = {t_stat_welch:.4f}, p = {p_welch:.4f}")

# 方差齐性检验
stat_levene, p_levene = stats.levene(group1, group2)
print(f"\n方差齐性检验 (Levene):")
print(f"  统计量: {stat_levene:.4f}")
print(f"  p值: {p_levene:.4f}")
print(f"  结论: {'方差不齐' if p_levene < 0.05 else '方差齐性假设成立'}")

### 4.3 独立双样本t检验可视化

In [ ]:
# 使用方法A和方法B的数据
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 箱线图
data = {'方法A': group_A, '方法B': group_B}
bp = axes[0].boxplot([data['方法A'], data['方法B']], labels=['方法A', '方法B'],
                     patch_artist=True, widths=0.6)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightcoral')
axes[0].set_ylabel('分数')
axes[0].set_title('两种方法分数分布')
axes[0].grid(True, alpha=0.3, axis='y')

# 添加均值标记
means = [np.mean(group_A), np.mean(group_B)]
axes[0].plot([1, 2], means, 'ro', markersize=8, label='均值')
axes[0].legend()

# 小提琴图
positions = [1, 2]
parts = axes[1].violinplot([data['方法A'], data['方法B']], positions=positions,
                          showmeans=True, showmedians=True, showextrema=True)
parts['bodies'][0].set_facecolor('lightblue')
parts['bodies'][1].set_facecolor('lightcoral')
axes[1].set_xticks([1, 2])
axes[1].set_xticklabels(['方法A', '方法B'])
axes[1].set_ylabel('分数')
axes[1].set_title('分数分布密度')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"方法A均值 ± 标准误: {np.mean(group_A):.2f} ± {stats.sem(group_A):.2f}")
print(f"方法B均值 ± 标准误: {np.mean(group_B):.2f} ± {stats.sem(group_B):.2f}")
print(f"均值差: {np.mean(group_A) - np.mean(group_B):.2f}")

## 5. 配对样本t检验

### 5.1 检验示例

测量20人减肥前后的体重变化，检验减肥效果是否显著。

In [ ]:
# 生成配对样本数据
np.random.seed(42)
n = 20

# 减肥前的体重
before = np.random.normal(75, 5, n)

# 减肥后的体重（平均减少2公斤）
after = before - np.random.normal(2, 1, n)

# 执行配对t检验
t_stat, p_value = stats.ttest_rel(after, before)

# 计算差值
differences = after - before
mean_diff = np.mean(differences)
std_diff = np.std(differences, ddof=1)

print("配对样本t检验结果")
print("=" * 40)
print(f"减肥前 - 均值: {np.mean(before):.2f} kg, 标准差: {np.std(before, ddof=1):.2f}")
print(f"减肥后 - 均值: {np.mean(after):.2f} kg, 标准差: {np.std(after, ddof=1):.2f}")
print(f"平均变化: {mean_diff:.2f} kg")
print(f"变化的标准差: {std_diff:.2f}")
print(f"\nt统计量: {t_stat:.4f}")
print(f"p值: {p_value:.4f}")

# 决策
alpha = 0.05
if p_value < alpha:
    print(f"\n结论：拒绝原假设，减肥效果显著")
else:
    print(f"\n结论：不拒绝原假设，没有证据表明减肥效果显著")

### 5.2 配对样本可视化

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 左上：配对散点图
axes[0, 0].scatter(before, after, alpha=0.6, s=80)
# 添加对角线
min_val = min(np.min(before), np.min(after))
max_val = max(np.max(before), np.max(after))
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=2, label='无变化线')
axes[0, 0].set_xlabel('减肥前体重 (kg)')
axes[0, 0].set_ylabel('减肥后体重 (kg)')
axes[0, 0].set_title('配对散点图')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 右上：差值分布
axes[0, 1].hist(differences, bins=10, edgecolor='black', alpha=0.7, color='lightgreen')
axes[0, 1].axvline(mean_diff, color='red', linewidth=2, linestyle='--', label=f'平均变化 = {mean_diff:.2f} kg')
axes[0, 1].axvline(0, color='black', linewidth=2, linestyle='-', label='无变化')
axes[0, 1].set_xlabel('体重变化 (kg)')
axes[0, 1].set_ylabel('频数')
axes[0, 1].set_title('体重变化分布')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 左下：前后对比箱线图
bp = axes[1, 0].boxplot([before, after], labels=['减肥前', '减肥后'],
                     patch_artist=True, widths=0.6)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightgreen')
axes[1, 0].set_ylabel('体重 (kg)')
axes[1, 0].set_title('减肥前后体重对比')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 右下：个体变化折线图
subjects = np.arange(1, n+1)
axes[1, 1].plot(subjects, before, 'bo-', label='减肥前', markersize=6, alpha=0.7)
axes[1, 1].plot(subjects, after, 'go-', label='减肥后', markersize=6, alpha=0.7)
for i in range(n):
    axes[1, 1].plot([i+1, i+1], [before[i], after[i]], 'gray', alpha=0.5, linewidth=1)
axes[1, 1].set_xlabel('个体')
axes[1, 1].set_ylabel('体重 (kg)')
axes[1, 1].set_title('个体体重变化')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 效应量计算（Cohen's d）

Cohen's d 是衡量效应大小的重要指标，表示两组均值差相对于标准差的大小。

In [ ]:
def cohens_d(group1, group2):
    """计算 Cohen's d（等方差假设）"""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    
    # 合并标准差
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1 + n2 - 2))
    
    # Cohen's d
    d = (np.mean(group1) - np.mean(group2)) / pooled_std
    
    return d

def cohens_d_welch(group1, group2):
    """计算 Cohen's d（异方差假设）"""
    n1, n2 = len(group1), len(group2)
    
    # 分离标准差
    std1, std2 = np.std(group1, ddof=1), np.std(group2, ddof=1)
    se1, se2 = std1 / np.sqrt(n1), std2 / np.sqrt(n2)
    se_diff = np.sqrt(se1**2 + se2**2)
    
    # Cohen's d
    d = (np.mean(group1) - np.mean(group2)) / se_diff * np.sqrt(2)
    
    return d

def interpret_cohens_d(d):
    """解释 Cohen's d 的大小"""
    abs_d = abs(d)
    if abs_d < 0.2:
        return "小效应"
    elif abs_d < 0.5:
        return "中等效应"
    elif abs_d < 0.8:
        return "大效应"
    else:
        return "极大效应"

# 计算各组Cohen's d
d_AB = cohens_d(group_A, group_B)
d_before_after = cohens_d_welch(after, before)

print("效应量（Cohen's d）")
print("=" * 40)
print(f"\n方法A vs 方法B: d = {d_AB:.3f}")
print(f"解释: {interpret_cohens_d(d_AB)}")

print(f"\n减肥前 vs 减肥后: d = {d_before_after:.3f}")
print(f"解释: {interpret_cohens_d(d_before_after)}")

print("\nCohen's d 解释标准：")
print("  |d| < 0.2: 小效应")
print("  0.2 ≤ |d| < 0.5: 中等效应")
print("  0.5 ≤ |d| < 0.8: 大效应")
print("  |d| ≥ 0.8: 极大效应")

## 7. 假设检验

In [ ]:
# 1. 正态性检验 (Shapiro-Wilk)
print("1. 正态性检验 (Shapiro-Wilk)")
print("=" * 50)

def check_normality(data, name):
    """检查数据的正态性"""
    stat, p = stats.shapiro(data)
    print(f"{name}:")
    print(f"  Shapiro-Wilk 统计量: {stat:.4f}")
    print(f"  p值: {p:.4f}")
    print(f"  结论: {'拒绝正态性假设' if p < 0.05 else '无法拒绝正态性假设'}")
    print()

check_normality(group_A, "方法A")
check_normality(group_B, "方法B")
check_normality(before, "减肥前")
check_normality(after, "减肥后")
check_normality(differences, "体重变化")

# 2. 方差齐性检验
print("\n2. 方差齐性检验")
print("=" * 50)

# Levene 检验
stat, p = stats.levene(group_A, group_B)
print("Levene 检验 (方法A vs 方法B):")
print(f"  统计量: {stat:.4f}")
print(f"  p值: {p:.4f}")
print(f"  结论: {'方差不齐' if p < 0.05 else '方差齐性假设成立'}")

# Bartlett 检验（需要正态性）
stat, p = stats.bartlett(group_A, group_B)
print(f"\nBartlett 检验 (方法A vs 方法B):")
print(f"  统计量: {stat:.4f}")
print(f"  p值: {p:.4f}")
print(f"  结论: {'方差不齐' if p < 0.05 else '方差齐性假设成立'}")

## 8. Q-Q 图（正态性可视化）

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 设置数据列表
datasets = [group_A, group_B, before, after, differences, sample]
titles = ['方法A', '方法B', '减肥前', '减肥后', '体重变化', '电池寿命']

for i, (data, title) in enumerate(zip(datasets, titles)):
    row, col = i // 3, i % 3
    
    stats.probplot(data, dist="norm", plot=axes[row, col])
    axes[row, col].set_title(f'{title} Q-Q图')
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Q-Q图解释：")
print("- 如果数据点大致沿着直线分布，说明数据符合正态分布")
print("- 如果数据点偏离直线，特别是尾部偏离，说明数据偏离正态分布")

## 9. 不同样本量对检验的影响

In [ ]:
# 模拟不同样本量的检验
np.random.seed(42)
sample_sizes = [5, 10, 20, 30, 50, 100, 200]
effect_size = 0.5  # 中等效应
n_simulations = 1000

results = []

for n in sample_sizes:
    significant_count = 0
    
    for _ in range(n_simulations):
        # 生成两组数据
        group1 = np.random.normal(0, 1, n)
        group2 = np.random.normal(effect_size, 1, n)
        
        # 执行t检验
        _, p_value = stats.ttest_ind(group1, group2)
        
        if p_value < 0.05:
            significant_count += 1
    
    power = significant_count / n_simulations
    results.append({'n': n, 'power': power})

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：功效曲线
n_values = [r['n'] for r in results]
powers = [r['power'] for r in results]

axes[0].plot(n_values, powers, 'bo-', linewidth=2, markersize=8)
axes[0].axhline(y=0.8, color='r', linestyle='--', linewidth=2, label='标准功效 0.8')
axes[0].set_xlabel('样本量')
axes[0].set_ylabel('统计功效')
axes[0].set_title(f'统计功效随样本量的变化 (效应量 = {effect_size})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 右图：p值分布
np.random.seed(42)
n_test = 30
effect_sizes = [0, 0.2, 0.5, 0.8]

for es in effect_sizes:
    p_values = []
    for _ in range(500):
        group1 = np.random.normal(0, 1, n_test)
        group2 = np.random.normal(es, 1, n_test)
        _, p_value = stats.ttest_ind(group1, group2)
        p_values.append(p_value)
    
    # 累积分布
    p_sorted = sorted(p_values)
    y = np.arange(1, len(p_sorted) + 1) / len(p_sorted)
    label = f'H₀为真' if es == 0 else f'效应量 = {es}'
    axes[1].plot(p_sorted, y, label=label, linewidth=2)

axes[1].set_xlabel('p值')
axes[1].set_ylabel('累积概率')
axes[1].set_title('不同效应量的p值分布')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察结果：")
print("1. 样本量越大，统计功效越高")
print("2. 效应量越大，p值越小（越显著）")
print("3. 当H₀为真时，p值均匀分布在[0,1]区间")

## 10. 总结

### t检验类型

| 检验类型 | 用途 | 假设 |
|---------|------|------|
| 单样本t检验 | 检验样本均值是否等于已知值 | 数据正态分布，观测独立 |
| 独立双样本t检验 | 比较两组独立样本的均值 | 两组独立，各组正态，方差相等（或使用Welch） |
| 配对样本t检验 | 比较配对样本的差值均值 | 差值正态分布，配对独立 |

### 关键要点

1. **前提条件检查：**
   - 正态性：Q-Q图、Shapiro-Wilk检验
   - 方差齐性：Levene检验、Bartlett检验
   - 独立性：通过研究设计保证

2. **选择合适的检验：**
   - 等方差：合并方差t检验
   - 异方差：Welch's t检验
   - 配对数据：配对t检验

3. **报告完整信息：**
   - t统计量和p值
   - 置信区间
   - 效应量（Cohen's d）
   - 假设检验结果

4. **可视化：**
   - 箱线图、小提琴图
   - 配对散点图、差值分布
   - Q-Q图
   - t分布和p值区域

5. **实际意义：**
   - 统计显著性 ≠ 实际重要性
   - 考虑效应量
   - 结合领域知识解释结果